# ML-SAD — original exploratory notebook (outputs removed)

**Note.** This notebook is preserved for transparency as the original exploratory
analysis. Its cell *outputs have been stripped* because they printed raw,
identifying records (email addresses, etc.). It also reads the private raw JSON
files, which are **not** distributed.

For the maintained, de-identified, reproducible pipeline use the `scripts/` and
`src/ml_sad/` package, which run on the public `data/` tables. This notebook is
reference-only and is not guaranteed to run as-is.


In [ ]:
import pandas as pd
import json

In [ ]:
import json
filepath = 'socialanxietyusers.json'
data = []
with open(filepath, 'r') as f:
    for line in f:
        # print(line)
        # Each line is a separate JSON object, so we load it individually
        data.append(line)
print(data)
print(len(data))

In [ ]:
import json

def strip_ipgeolocation(line: str) -> str:
    """
    Remove the "ipgeolocation": { … } field (and any trailing comma)
    from a single JSON-Lines string before loading.
    """
    key = '"ipgeolocation"'
    idx = line.find(key)
    if idx == -1:
        return line  # nothing to strip

    # find the opening brace of the ipgeolocation object
    colon = line.find(':', idx + len(key))
    start = line.find('{', colon)
    if start == -1:
        return line

    # walk forward to find the matching closing brace
    brace_count = 0
    for i, ch in enumerate(line[start:], start=start):
        if ch == '{':
            brace_count += 1
        elif ch == '}':
            brace_count -= 1
            if brace_count == 0:
                end = i + 1
                break
    else:
        return line  # unmatched braces

    # drop any comma immediately following the ipgeolocation object
    j = end
    while j < len(line) and line[j].isspace():
        j += 1
    if j < len(line) and line[j] == ',':
        end = j + 1

    # remove the field slice
    before = line[:idx].rstrip()
    after  = line[end:]
    # clean up stray commas
    if before.endswith(','):
        before = before[:-1]
    if after.lstrip().startswith(','):
        after = after.lstrip()[1:]
    # print(before+after)
    return before + after

input_path  = 'socialanxietyusers.json'
output_path = 'socialanxietyusers1.json'

with open(input_path,  'r', encoding='utf-8') as fin, \
     open(output_path, 'w', encoding='utf-8') as fout:

    for raw in fin:
        line = raw.rstrip('\n')
        clean = strip_ipgeolocation(line)
        # now it's safe to json.loads(clean)
        obj = json.loads(clean)
        fout.write(json.dumps(obj, ensure_ascii=False) + '\n')

print(f"Stripped file written to {output_path}")


In [ ]:
import json

filepath = 'socialanxietyusers1.json'
data = []

with open(filepath, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        try:
            record = json.loads(line)
        except json.JSONDecodeError:
            
            continue
        print(line)
        data.append(record)


# with open(filepath, 'r') as f:
#     for line in f:
#         line = line.strip(r'\n')
#         if not line:
#             continue
#         try:
#             record = json.loads(line)
#         except json.JSONDecodeError:
#             # wenn eine Zeile kein gültiges JSON ist, überspringen
#             continue
#         data.append(record)

# Jetzt enthält `data` eine Liste von Dictionaries, eines pro Zeile.

# Optional: in ein einziges Dictionary umwandeln, z. B. mit Zeilennummern als Schlüssel
data_dict = {i: rec for i, rec in enumerate(data)}
print(data_dict)
print(len(data_dict))
# Oder, falls jedes Objekt ein eindeutiges Feld wie 'user_id' hat:
# data_dict = {rec['user_id']: rec for rec in data}


In [ ]:
data_dict[100]['id']

In [ ]:
spin_data = []
for i in range(len(data_dict)):
    rec = data_dict[i]
    # print(rec)

    if 'spinquestionnaire' in rec and 'questionnaireScore' in rec['spinquestionnaire']:
        # score = rec['spinquestionnaire']['questionnaireScore']
        spin_data.append(data_dict[i])

# for i in range(len(spin_data)):
#     spin = spin_data[i]

#     if 'spinquestionnaire' in spin and 'questionnaireScore' in spin['spinquestionnaire']:
#         score = spin['spinquestionnaire']['questionnaireScore']
len(spin_data)


In [ ]:
spin_data = []
for i in range(len(data_dict)):
    rec = data_dict[i]
    # print(rec)

    if 'spinquestionnaire' in rec:
        print(rec['spinquestionnaire'])

In [ ]:
# spin_data = []
# for i in range(len(data_dict)):
#     rec = data_dict[i]
#     # print(rec)

#     if 'spinquestionnaire' in rec:
#         print(rec['spinquestionnaire'])

#         if rec['spinquestionnaire']['questions']:
#             questions = rec['spinquestionnaire']['questions']
#         else:
#             questions = rec['spinquestionnaire']['question1']+rec['spinquestionnaire']['question2']+rec['spinquestionnaire']['question3']+rec['spinquestionnaire']['question4']+rec['spinquestionnaire']['question5']
#             +rec['spinquestionnaire']['question6']+rec['spinquestionnaire']['question7']+rec['spinquestionnaire']['question8']+rec['spinquestionnaire']['question9']+rec['spinquestionnaire']['question10']+rec['spinquestionnaire']['question11']
#             +rec['spinquestionnaire']['question12']+rec['spinquestionnaire']['question13']+rec['spinquestionnaire']['question14']+rec['spinquestionnaire']['question15']+rec['spinquestionnaire']['question16']+rec['spinquestionnaire']['question17']
            
#         total = sum(int(answer) for answer in questions.values())
#         print(total)

   

In [ ]:
total_ques = []
severity_labels = []

for rec in data_dict.values():
    spin = rec.get('spinquestionnaire')
    if not spin:
        continue

    # Fragen extrahieren (entweder als Dict oder aus Einzel-Keys)
    if isinstance(spin.get('questions'), dict):
        questions = spin['questions']
    else:
        questions = {
            f'question{i}': spin.get(f'question{i}', 0)
            for i in range(1, 18)
    }

    # Summe aller Antworten
    total = sum(int(v) for v in questions.values())

    if 0 <= total <= 20:
        label = 0
    elif 21 <= total <= 30:
        label = 1
    elif 31 <= total <= 40:
        label = 2
    elif 41 <= total <= 50:
        label = 3
    elif 51 <= total <= 68:
        label = 4
    else:
        label = "Out of range"

    severity_labels.append(total)

    

print(severity_labels)


0-20: Minimal or no social anxiety; 21-30: Mild; 31-40: Moderate; 41-50: Severe; and 51-68: Very severe

In [ ]:
print(data_dict)

In [ ]:
suppression = []
avoidance = []
reappraisal = []
label = []


for i in range(1, 129):
    spin = data_dict[i]
    suppression.append(spin["supression_sum"])
    avoidance.append(spin["avoidance_sum"])
    reappraisal.append(spin["reappraisal_sum"])
    label.append(spin["label"])

print(severity_labels)
print(suppression)
print(avoidance)
print(reappraisal)
print(label)

data_new = {
    "severity_labels":    severity_labels,
    "suppression": suppression,
    "avoidance":   avoidance,
    "reappraisal": reappraisal,
    "label":       label
}




In [ ]:
df = pd.DataFrame()
df['severity_labels'] = 0
df['suppression_sum'] = 0
df['avoidance_sum'] = 0
df['reappraisal_sum'] = 0
df['label'] = 0

df['severity_labels'] = severity_labels
df['suppression_sum'] = suppression
df['avoidance_sum'] = avoidance
df['reappraisal_sum'] = reappraisal
df['label'] = label

# df['severity_labels']  = (df['severity_labels'] - df['severity_labels'].mean())/df['severity_labels'].std()
# df['suppression_sum']  = (df['suppression_sum'] - df['suppression_sum'].mean())/df['suppression_sum'].std()
# df['avoidance_sum']  = (df['avoidance_sum'] - df['avoidance_sum'].mean())/df['avoidance_sum'].std()
# df['reappraisal_sum']  = (df['reappraisal_sum'] - df['reappraisal_sum'].mean())/df['reappraisal_sum'].std()

# cor_sup = df['suppression_sum'].corr(df['severity_labels'])
# cor_avo = df['avoidance_sum'].corr(df['severity_labels'])
# cor_reap = df['reappraisal_sum'].corr(df['severity_labels'])
# print('sup', cor_sup)
# print('avo', cor_avo)
# print('reap', cor_reap)

In [ ]:
print(df['label'], df['severity_labels'])
df['label'].corr(df['severity_labels'], method='spearman')


In [ ]:
# df.corr(method='spearman')
df.corr()

In [ ]:
df_sad = df.loc[df['label'] == 1]
df_nosad = df.loc[df['label'] == 0]

In [ ]:
print(df_sad)
print(df_nosad)

In [ ]:
# cor_sup = df_sad['suppression_sum'].corr(df_sad['severity_labels'])
# cor_avo = df_sad['avoidance_sum'].corr(df_sad['severity_labels'])
# cor_reap = df_sad['reappraisal_sum'].corr(df_sad['severity_labels'])
# print('sup', cor_sup)
# print('avo', cor_avo)
# print('reap', cor_reap)

In [ ]:
# df_sad.corr(method='spearman')
df_sad.corr()

In [ ]:
df_nosad.corr()